# Generative Models — Exercises

Companion to the note [Generative Models](Generative%20Models.md).

Practise the core maths of the generative families in the note: the ELBO and Gaussian KL of VAEs, the reparameterization trick, the GAN minimax game and its losses, change of variables for flows, perplexity of autoregressive models, the diffusion forward process and FID. All implemented from scratch in NumPy and checked against SciPy/sklearn or Monte Carlo.

**16 exercises** · 🧠 Concept ×3 · ✍️ By hand ×6 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy import stats
from scipy.special import expit, softplus, logsumexp
from scipy.linalg import sqrtm
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
rng = np.random.default_rng(3)

def kl_gauss_full(mu1, S1, mu2, S2):
    """Reference KL( N(mu1,S1) || N(mu2,S2) ) for full covariance matrices."""
    k = len(mu1); S2i = np.linalg.inv(S2); dm = mu2 - mu1
    return 0.5 * (np.trace(S2i @ S1) + dm @ S2i @ dm - k + np.linalg.slogdet(S2)[1] - np.linalg.slogdet(S1)[1])

## Part A · Concepts

### Exercise 1 · Choosing a generative family
*🧠 Concept · ★☆☆*

Using the note's table, pick a family for each requirement and justify it in one line:

1. Exact log-likelihood of a test image. 2. Fastest single-pass sampling of sharp images. 3. State-of-the-art image quality, sampling time not critical.
4. Text generation. 5. A compressed representation for [[Anomaly Detection]].

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Flows and autoregressive models give exact likelihoods; GANs sample in one forward pass; diffusion needs many denoising steps.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Normalizing flow** (or autoregressive): exact $\log p(x)$ via change of variables. VAEs only give a lower bound (ELBO).
2. **GAN**: one generator forward pass, sharp samples (but unstable training).
3. **Diffusion**: best FID today, but sampling needs tens to thousands of network evaluations.
4. **Autoregressive** Transformer: $p(x)=\prod_t p(x_t\mid x_{<t})$ is natural for sequences.
5. **Autoencoder**: high reconstruction error flags inputs unlike the training data.

</details>

### Exercise 2 · Blurry VAEs, collapsing GANs
*🧠 Concept · ★★☆*

1. Why are VAE samples typically blurry? Relate it to the Gaussian likelihood $p(x\mid z)$ and to the direction of the KL in the ELBO.
2. What is **mode collapse** in GANs, and why does the minimax objective allow it? Name two remedies.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A Gaussian decoder's log-likelihood is an MSE; MSE is minimised by the *mean* of the plausible outputs.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. With $p(x\mid z)=\mathcal N(\mu_\theta(z),\sigma^2 I)$, $\log p$ is a scaled MSE; when $z$ does not pin down the image exactly,
   the MSE-optimal output is the **average** of the plausible images, which is blurry. Maximum likelihood (forward KL $D_{KL}(p_{data}\Vert p_\theta)$)
   is also *mass-covering*: the model must put probability everywhere the data is, and spreads mass between modes.
2. The generator maps many $z$ to the same few outputs that currently fool $D$. The generator is only optimised against the *current* $D$ and
   is not penalised for missing modes (the JS objective is mode-seeking in practice). Remedies: Wasserstein loss/gradient penalty,
   minibatch discrimination, unrolled GANs, spectral normalisation, or switching to diffusion.

</details>

### Exercise 3 · Autoencoder vs generative model
*🧠 Concept · ★☆☆*

A plain autoencoder learns an encoder $z = e(x)$ and decoder $\hat x = d(z)$. Why can you not simply sample $z\sim\mathcal N(0,I)$ and decode
to get realistic images? What exactly does the VAE add to fix this?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

What does the latent space of a plain AE look like away from the training codes?

</details>

<details>
<summary><b>✅ Solution</b></summary>

A plain AE puts codes wherever reconstruction is easiest: the codes can be scattered, with large "holes" the decoder never saw, and there is
no prior to sample from. Decoding a random $z$ gives garbage. The VAE (i) makes the encoder output a **distribution** $q(z\mid x)$, (ii) adds
$D_{KL}(q(z\mid x)\Vert p(z))$ which pulls all posteriors towards the prior $\mathcal N(0,I)$ so the latent space is filled smoothly, and
(iii) trains the decoder on samples of $z$ (reparameterization), so decoding prior samples works.

</details>

## Part B · By hand

### Exercise 4 · Gaussian KL in closed form
*✍️ By hand · ★★☆*

Derive $D_{KL}\big(\mathcal N(\mu,\mathrm{diag}\,\sigma^2)\,\Vert\,\mathcal N(0,I)\big) = \tfrac12\sum_j\big(\sigma_j^2+\mu_j^2-1-\log\sigma_j^2\big)$
(one dimension suffices, then sum). Evaluate it for $\mu = (1, 0)$, $\sigma = (1, 0.5)$.

In [ ]:
kl_hand = None

check('KL', kl_hand, kl_gauss_full(np.array([1., 0]), np.diag([1, 0.25]), np.zeros(2), np.eye(2)), tol=1e-4)

<details>
<summary><b>💡 Hint 1</b></summary>

$D_{KL} = \mathbb E_q[\log q(z) - \log p(z)]$ with $\log q = -\tfrac12\log 2\pi\sigma^2 - \tfrac{(z-\mu)^2}{2\sigma^2}$ and $\log p = -\tfrac12\log2\pi - \tfrac{z^2}{2}$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Use $\mathbb E_q[(z-\mu)^2]=\sigma^2$ and $\mathbb E_q[z^2] = \sigma^2+\mu^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathbb E_q[\log q - \log p] = -\tfrac12\log\sigma^2 - \tfrac12 + \tfrac12(\sigma^2+\mu^2)$, i.e. $\tfrac12(\sigma^2+\mu^2-1-\log\sigma^2)$ per dimension.
Dimension 1: $\tfrac12(1+1-1-0) = 0.5$. Dimension 2: $\tfrac12(0.25 - 1 - \log 0.25) = \tfrac12(0.6363) = 0.3181$. Total $\approx 0.8181$.

```python
kl_hand = 0.5 * ((1 + 1 - 1 - 0) + (0.25 + 0 - 1 - np.log(0.25)))
```

</details>

### Exercise 5 · Where the ELBO comes from
*✍️ By hand · ★★☆*

Show that for any distribution $q(z\mid x)$
$$\log p(x) = \underbrace{\mathbb E_q[\log p(x\mid z)] - D_{KL}(q(z\mid x)\Vert p(z))}_{\text{ELBO}} + D_{KL}(q(z\mid x)\Vert p(z\mid x)).$$
Conclude that the ELBO is a lower bound, and say when it is tight. (See also [[Variational Inference]].)

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Start from $D_{KL}(q(z\mid x)\Vert p(z\mid x)) = \mathbb E_q[\log q(z\mid x) - \log p(z\mid x)]$ and use Bayes: $p(z\mid x) = p(x\mid z)p(z)/p(x)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$D_{KL}(q\Vert p(z\mid x)) = \mathbb E_q[\log q - \log p(x\mid z) - \log p(z) + \log p(x)] = -\mathbb E_q[\log p(x\mid z)] + D_{KL}(q\Vert p(z)) + \log p(x)$.
Rearranging gives the identity. Since $D_{KL}\ge0$, $\log p(x)\ge\text{ELBO}$, with equality iff $q(z\mid x) = p(z\mid x)$.
So maximising the ELBO over $q$ both tightens the bound and makes $q$ approximate the true posterior; maximising it over $\theta$ approximately maximises the likelihood.

</details>

### Exercise 6 · The optimal discriminator
*✍️ By hand · ★★☆*

For fixed generator density $p_g$, the GAN value is $V(D) = \int p_{data}(x)\log D(x) + p_g(x)\log(1-D(x))\,dx$.

(a) Show the optimal discriminator is $D^*(x) = \frac{p_{data}(x)}{p_{data}(x)+p_g(x)}$ and evaluate it at a point where $p_{data}=0.6, p_g=0.2$.
(b) At the global optimum $p_g = p_{data}$, compute $V(D^*)$.

In [ ]:
D_star = None
V_opt = None

_D = np.linspace(1e-4, 1 - 1e-4, 200001)
check('D* (brute-force argmax)', D_star, _D[np.argmax(0.6 * np.log(_D) + 0.2 * np.log(1 - _D))], tol=1e-3)
check('V at equilibrium', V_opt, 2 * np.log(0.5), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Maximise $a\log D + b\log(1-D)$ pointwise in $D$: derivative $a/D - b/(1-D) = 0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $a/D = b/(1-D) \Rightarrow D^* = a/(a+b)$; here $0.6/0.8 = 0.75$.
(b) With $p_g=p_{data}$, $D^*\equiv \tfrac12$ and $V = \log\tfrac12 + \log\tfrac12 = -\log 4 \approx -1.386$.
In general $V(D^*) = -\log 4 + 2\,\mathrm{JSD}(p_{data}\Vert p_g)$, so the generator minimises a Jensen–Shannon divergence.

```python
D_star = 0.75
V_opt = -np.log(4)
```

</details>

### Exercise 7 · Change of variables (flows)
*✍️ By hand · ★☆☆*

A one-layer flow maps $z\sim\mathcal N(0,1)$ to $x = f(z) = 2z + 1$. Using $\log p_x(x) = \log p_z(f^{-1}(x)) + \log\left|\frac{d f^{-1}}{dx}\right|$,
compute $\log p_x(3)$.

In [ ]:
logp_3 = None

check('log p(3)', logp_3, stats.norm(1, 2).logpdf(3), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

$f^{-1}(3) = 1$ and $\frac{df^{-1}}{dx} = \tfrac12$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\log p_x(3) = \log\mathcal N(1;0,1) + \log\tfrac12 = (-\tfrac12\log2\pi - \tfrac12) - \log 2 = -0.9189 - 0.5 - 0.6931 = -2.1121$,
which equals $\log\mathcal N(3; 1, 2^2)$ as it should. Flows need invertible $f$ with a tractable Jacobian determinant.

```python
logp_3 = -0.5 * np.log(2 * np.pi) - 0.5 - np.log(2)
```

</details>

### Exercise 8 · Perplexity of an autoregressive model
*✍️ By hand · ★☆☆*

An autoregressive LM assigns probabilities $0.5, 0.25, 0.125, 0.5$ to the four tokens of a test sentence.
Compute the average negative log-likelihood (in nats) and the perplexity $\exp(\text{NLL})$.

In [ ]:
nll = None
ppl = None

_p = np.array([0.5, 0.25, 0.125, 0.5])
check('NLL', nll, -np.mean(np.log(_p)), tol=1e-4); check('perplexity', ppl, np.exp(-np.mean(np.log(_p))), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

In bits the per-token losses are 1, 2, 3, 1. Perplexity $= 2^{\text{mean bits}}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Mean bits $= 7/4 = 1.75$, so perplexity $= 2^{1.75} \approx 3.364$ and NLL $= 1.75\ln2 \approx 1.213$ nats.
Perplexity is the effective number of equally likely choices per token.

```python
nll = 1.75 * np.log(2)
ppl = 2 ** 1.75
```

</details>

### Exercise 9 · Diffusion: the forward process in closed form
*✍️ By hand · ★☆☆*

The forward process adds noise as $x_t = \sqrt{1-\beta_t}\,x_{t-1} + \sqrt{\beta_t}\,\varepsilon_t$. With $\bar\alpha_t = \prod_{s\le t}(1-\beta_s)$ this gives
$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$. For $\beta_1=\beta_2=\beta_3=0.1$ and $x_0 = 2$, compute $\bar\alpha_3$ and the mean and variance of $x_3\mid x_0$.

In [ ]:
abar3 = None
mean_x3 = None
var_x3 = None

_x = np.full(400000, 2.0)
for _ in range(3):
    _x = np.sqrt(0.9) * _x + np.sqrt(0.1) * rng.normal(size=_x.size)   # simulate step by step
check('abar_3', abar3, 0.9 ** 3)
check('mean (simulated)', mean_x3, _x.mean(), tol=5e-3); check('variance (simulated)', var_x3, _x.var(), tol=5e-3)

<details>
<summary><b>💡 Hint</b></summary>

$\bar\alpha_3 = 0.9^3$. Variances of independent Gaussian noise terms add after scaling.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\bar\alpha_3 = 0.729$, mean $\sqrt{0.729}\cdot2 = 0.8538\cdot2 = 1.7076$, variance $1-0.729 = 0.271$.
Because the composition of Gaussian steps is Gaussian, training can sample any $t$ directly without simulating the chain (Exercise 15).

```python
abar3 = 0.729
mean_x3 = 2 * np.sqrt(0.729)
var_x3 = 1 - 0.729
```

</details>

## Part C · Code from scratch

### Exercise 10 · Reparameterization vs score-function gradients
*💻 Code · ★★☆*

Let $z\sim\mathcal N(\mu,\sigma^2)$ and $J(\mu,\sigma) = \mathbb E[z^2] = \mu^2+\sigma^2$, so $\nabla J = (2\mu, 2\sigma)$.
With $\mu=1.5, \sigma=0.5$ and $N = 100{,}000$ samples:

1. **Pathwise** (reparameterization, $z=\mu+\sigma\varepsilon$): per-sample gradients $2z$ (w.r.t. $\mu$) and $2z\varepsilon$ (w.r.t. $\sigma$). Store their means in `grad_path`.
2. **Score function** (REINFORCE): per-sample gradient w.r.t. $\mu$ is $z^2\cdot\frac{z-\mu}{\sigma^2}$. Store the per-sample variances of both $\mu$-estimators in `var_path`, `var_score`.

In [ ]:
mu_rp, sig_rp, N_rp = 1.5, 0.5, 100_000
grad_path = None   # [d/dmu, d/dsigma]
var_path = None
var_score = None

check('pathwise gradient', grad_path, [2 * 1.5, 2 * 0.5], tol=0.03)
check('reparameterization has (much) lower variance', None if var_path is None else var_score > 5 * var_path, True)

<details>
<summary><b>💡 Hint</b></summary>

`eps = rng.normal(size=N_rp); z = mu_rp + sig_rp * eps`. $\partial z/\partial\mu = 1$, $\partial z/\partial\sigma = \varepsilon$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The pathwise estimator differentiates *through* the sample: $\nabla_\mu z^2 = 2z$, $\nabla_\sigma z^2 = 2z\varepsilon$. Its variance here is $4\sigma^2 = 1$,
while the score-function estimator has variance ≈ 55 (about 50× larger). This variance gap is why VAEs use $z = \mu+\sigma\odot\varepsilon$ (note), whereas REINFORCE is needed only when $f$ is not differentiable (e.g. discrete $z$, RL).

```python
mu_rp, sig_rp, N_rp = 1.5, 0.5, 100_000
eps = rng.normal(size=N_rp); z = mu_rp + sig_rp * eps
g_mu, g_sig = 2 * z, 2 * z * eps
g_score = z ** 2 * (z - mu_rp) / sig_rp ** 2
grad_path = [g_mu.mean(), g_sig.mean()]
var_path, var_score = g_mu.var(), g_score.var()
print(grad_path, g_score.mean(), var_path, var_score)
```

</details>

### Exercise 11 · VAE loss (negative ELBO)
*💻 Code · ★☆☆*

Implement for a batch:

- `kl_diag(mu, logvar)`: $\tfrac12\sum_j(e^{\ell_j}+\mu_j^2-1-\ell_j)$ per sample, with $\ell = \log\sigma^2$ (the parametrisation used in practice).
- `bce_sum(x, p)`: Bernoulli reconstruction loss $-\sum_j [x_j\log p_j + (1-x_j)\log(1-p_j)]$ per sample.
- `neg_elbo(x, p, mu, logvar)`: their sum, averaged over the batch.

In [ ]:
def kl_diag(mu, logvar):
    return None

def bce_sum(x, p):
    return None

def neg_elbo(x, p, mu, logvar):
    return None

mu_v, lv_v = rng.normal(size=(4, 3)), rng.normal(0, 0.5, size=(4, 3))
x_v = (rng.random((4, 10)) < 0.5).astype(float); p_v = rng.uniform(0.05, 0.95, size=(4, 10))

check('KL per sample', kl_diag(mu_v, lv_v), [kl_gauss_full(m, np.diag(np.exp(l)), np.zeros(3), np.eye(3)) for m, l in zip(mu_v, lv_v)])
check('BCE per sample', bce_sum(x_v, p_v), [log_loss(x, p, normalize=False, labels=[0, 1]) for x, p in zip(x_v, p_v)])
_ref = np.mean([log_loss(x, p, normalize=False, labels=[0, 1]) + kl_gauss_full(m, np.diag(np.exp(l)), np.zeros(3), np.eye(3)) for x, p, m, l in zip(x_v, p_v, mu_v, lv_v)])
check('negative ELBO', neg_elbo(x_v, p_v, mu_v, lv_v), _ref)

<details>
<summary><b>💡 Hint</b></summary>

Sum over axis 1 (features), average over axis 0 (batch) only in `neg_elbo`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Predicting $\log\sigma^2$ keeps $\sigma$ positive without constraints. The KL term acts as a regulariser; weighting it by $\beta>1$ gives the $\beta$-VAE
(more disentangled, blurrier), while $\beta\to0$ recovers a plain autoencoder.

```python
def kl_diag(mu, logvar):
    return 0.5 * np.sum(np.exp(logvar) + mu ** 2 - 1 - logvar, axis=1)

def bce_sum(x, p):
    return -np.sum(x * np.log(p) + (1 - x) * np.log(1 - p), axis=1)

def neg_elbo(x, p, mu, logvar):
    return np.mean(bce_sum(x, p) + kl_diag(mu, logvar))

mu_v, lv_v = rng.normal(size=(4, 3)), rng.normal(0, 0.5, size=(4, 3))
x_v = (rng.random((4, 10)) < 0.5).astype(float); p_v = rng.uniform(0.05, 0.95, size=(4, 10))
```

</details>

### Exercise 12 · GAN losses from logits, and why the non-saturating loss
*💻 Code · ★★☆*

Work with discriminator **logits** $s$ ($D=\sigma(s)$) and the numerically stable identity $-\log\sigma(s) = \mathrm{softplus}(-s) = \log(1+e^{-s})$.

1. `d_loss(s_real, s_fake)`: $-\mathbb E[\log D(x)] - \mathbb E[\log(1-D(G(z)))]$.
2. `g_loss_sat(s_fake)`: the original minimax generator loss $\mathbb E[\log(1-D(G(z)))]$.
3. `g_loss_ns(s_fake)`: the non-saturating loss $-\mathbb E[\log D(G(z))]$.

Then, early in training the discriminator wins: $s_{fake} = -5$. Compute $\partial/\partial s$ of both generator losses at $s=-5$ (finite differences are fine): `grad_sat`, `grad_ns`.

In [ ]:
def softplus_np(a):
    return np.logaddexp(0, a)          # stable log(1 + e^a)

def d_loss(s_real, s_fake):
    return None

def g_loss_sat(s_fake):
    return None

def g_loss_ns(s_fake):
    return None

s_r, s_f = rng.normal(1, 2, size=50), rng.normal(-1, 2, size=50)
grad_sat = None
grad_ns = None

_y = np.r_[np.ones(50), np.zeros(50)]
check('D loss = 2 x binary cross-entropy', d_loss(s_r, s_f), 2 * log_loss(_y, expit(np.r_[s_r, s_f])))
check('stable for huge logits', None if d_loss(s_r, s_f) is None else np.isfinite(d_loss(np.array([1000.]), np.array([-1000.]))), True)
check('saturating gradient ~ -sigma(-5)... tiny', grad_sat, -expit(-5), tol=1e-5)
check('non-saturating gradient ~ -1', grad_ns, -expit(5), tol=1e-5)

<details>
<summary><b>💡 Hint 1</b></summary>

$-\log(1-\sigma(s)) = \mathrm{softplus}(s)$, so $\log(1-D) = -\mathrm{softplus}(s)$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$\frac{d}{ds}[-\mathrm{softplus}(s)] = -\sigma(s)$ and $\frac{d}{ds}\mathrm{softplus}(-s) = -\sigma(-s)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

At $s=-5$: saturating gradient $-\sigma(-5) \approx -0.0067$, non-saturating $-\sigma(5)\approx -0.9933$, i.e. about 150× larger.
When $D$ confidently rejects fakes, $\log(1-D)$ is flat, so the minimax generator learns nothing; $-\log D$ has the same fixed point
but strong gradients exactly when the generator is bad. This is the loss Goodfellow et al. actually used.

```python
def softplus_np(a):
    return np.logaddexp(0, a)

def d_loss(s_real, s_fake):
    return np.mean(softplus_np(-s_real)) + np.mean(softplus_np(s_fake))

def g_loss_sat(s_fake):
    return np.mean(-softplus_np(s_fake))

def g_loss_ns(s_fake):
    return np.mean(softplus_np(-s_fake))

s_r, s_f = rng.normal(1, 2, size=50), rng.normal(-1, 2, size=50)
h = 1e-6
grad_sat = (g_loss_sat(np.array([-5 + h])) - g_loss_sat(np.array([-5 - h]))) / (2 * h)
grad_ns = (g_loss_ns(np.array([-5 + h])) - g_loss_ns(np.array([-5 - h]))) / (2 * h)
print(grad_sat, grad_ns)
```

</details>

### Exercise 13 · A linear autoencoder learns PCA
*💻 Code · ★★☆*

Train a linear autoencoder $\hat x = x E D$ with $E\in\mathbb R^{5\times2}, D\in\mathbb R^{2\times5}$ by gradient descent on the MSE of centred data `X_ae`
(lr 0.01, 3000 steps, small random init). Store the final reconstruction MSE in `mse_ae`. It should match PCA with 2 components.
Does $E$ equal the PCA loadings?

In [ ]:
X_ae = rng.normal(size=(300, 5)) * np.array([3, 2, .5, .3, .2]) @ np.linalg.qr(rng.normal(size=(5, 5)))[0]
Xc_ae = X_ae - X_ae.mean(0)
mse_ae = None

_p = PCA(2).fit(X_ae)
check('linear AE reaches PCA error', mse_ae, np.mean((_p.inverse_transform(_p.transform(X_ae)) - X_ae) ** 2), tol=1e-4)

<details>
<summary><b>💡 Hint 1</b></summary>

With $Z = X_cE$ and $R = ZD - X_c$: $\nabla_D = \frac2{N}Z^\top R$ (divide additionally by 5 if you average over features too; it only rescales the step).

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$\nabla_E = \frac2N X_c^\top (R D^\top)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

A linear AE with MSE loss reaches the same optimum as PCA (Baldi & Hornik, 1989): its decoder spans the top-2 principal subspace.
But $E$ and $D$ are only determined up to an invertible $2\times2$ matrix ($ED = EAA^{-1}D$), so they are generally **not** the orthonormal PCA loadings.
Non-linear AEs generalise PCA to curved manifolds.

```python
X_ae = rng.normal(size=(300, 5)) * np.array([3, 2, .5, .3, .2]) @ np.linalg.qr(rng.normal(size=(5, 5)))[0]
Xc_ae = X_ae - X_ae.mean(0)
E = rng.normal(0, 0.1, size=(5, 2)); D = rng.normal(0, 0.1, size=(2, 5))
for _ in range(3000):
    Z = Xc_ae @ E; R = Z @ D - Xc_ae
    gD = 2 * Z.T @ R / len(Xc_ae); gE = 2 * Xc_ae.T @ (R @ D.T) / len(Xc_ae)
    D -= 0.01 * gD; E -= 0.01 * gE
mse_ae = np.mean((Xc_ae @ E @ D - Xc_ae) ** 2)
print(mse_ae)
```

</details>

### Exercise 14 · An affine normalizing flow trained by maximum likelihood
*💻 Code · ★★☆*

Model $x = a z + b$ with $z\sim\mathcal N(0,1)$ and parameters $s = \log a$, $b$.

1. Implement `flow_loglik(x, s, b)` $= \sum_i [\log\mathcal N(u_i;0,1) - s]$ with $u_i = (x_i - b)e^{-s}$.
2. Fit $(s, b)$ by gradient ascent on the mean log-likelihood using $\partial/\partial b = \overline{u}\,e^{-s}$ and $\partial/\partial s = \overline{u^2} - 1$.
Store the fitted `a_fit = exp(s)` and `b_fit`.

In [ ]:
def flow_loglik(x, s, b):
    return None

x_fl = rng.normal(4.0, 1.5, size=2000)
a_fit = None
b_fit = None

check('log-likelihood = scipy', flow_loglik(x_fl, np.log(2.0), 1.0), stats.norm(1.0, 2.0).logpdf(x_fl).sum())
check('a = MLE std', a_fit, x_fl.std(), tol=1e-3); check('b = MLE mean', b_fit, x_fl.mean(), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Start at $s=0, b=0$ with learning rate 0.1 for a few thousand steps. The $-s$ term is the log-determinant of the inverse map.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The maximum-likelihood affine flow recovers the Gaussian MLE: $b=\bar x$, $a = $ the (biased) standard deviation. Real flows (RealNVP, Glow)
stack many invertible layers whose Jacobian determinants stay cheap (triangular), and are trained with exactly this objective.

```python
def flow_loglik(x, s, b):
    u = (x - b) * np.exp(-s)
    return np.sum(-0.5 * u ** 2 - 0.5 * np.log(2 * np.pi) - s)

x_fl = rng.normal(4.0, 1.5, size=2000)
s, b = 0.0, 0.0
for _ in range(5000):
    u = (x_fl - b) * np.exp(-s)
    b += 0.1 * u.mean() * np.exp(-s)
    s += 0.1 * (np.mean(u ** 2) - 1)
a_fit, b_fit = np.exp(s), b
print(a_fit, b_fit)
```

</details>

### Exercise 15 · Diffusion: sampling x_t and the optimal noise predictor
*💻 Code · ★★★*

DDPM trains a network $\varepsilon_\theta(x_t, t)$ to predict the noise. For 1-D data $x_0\sim\mathcal N(0, s^2)$ with $s=2$, the best possible predictor is linear:
$\mathbb E[\varepsilon\mid x_t] = \frac{\sqrt{1-\bar\alpha_t}}{\bar\alpha_t s^2 + 1-\bar\alpha_t}\,x_t$.

1. With the linear schedule $\beta_t$ = `np.linspace(1e-4, 0.02, 1000)`, compute `abar` ($\bar\alpha_t$ for all $t$).
2. Implement `q_sample(x0, t, eps)`.
3. For $t = 300$, draw 200k pairs $(x_0,\varepsilon)$, form $x_t$, and fit the least-squares slope of $\varepsilon$ on $x_t$ (no intercept): `slope_fit`.

In [ ]:
betas = np.linspace(1e-4, 0.02, 1000)
abar = None

def q_sample(x0, t, eps):
    return None

slope_fit = None

check('abar[0] = 1 - beta_1', None if abar is None else abar[0], 1 - 1e-4)
check('abar is the running product', None if abar is None else abar[999], np.exp(np.sum(np.log(1 - betas))))
_ab = np.exp(np.sum(np.log(1 - betas[:301])))
check('slope = Bayes-optimal denoiser', slope_fit, np.sqrt(1 - _ab) / (_ab * 4 + 1 - _ab), tol=0.01)

<details>
<summary><b>💡 Hint 1</b></summary>

`abar = np.cumprod(1 - betas)`; use index `t` for $\bar\alpha$ (0-based).

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Slope without intercept: `np.sum(eps * xt) / np.sum(xt ** 2)`. Why is it the formula above? $\mathrm{Cov}(\varepsilon, x_t) = \sqrt{1-\bar\alpha}$, $\mathrm{Var}(x_t) = \bar\alpha s^2 + 1-\bar\alpha$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

For jointly Gaussian variables, $\mathbb E[\varepsilon\mid x_t] = \frac{\mathrm{Cov}(\varepsilon,x_t)}{\mathrm{Var}(x_t)}x_t$. The regression recovers it.
Note `abar[-1]` $\approx 4\times10^{-5}$: after 1000 steps the data is essentially pure noise, so sampling can start from $\mathcal N(0,1)$.
Predicting $\varepsilon$ is equivalent to estimating the score $\nabla_{x_t}\log p(x_t) = -\mathbb E[\varepsilon\mid x_t]/\sqrt{1-\bar\alpha_t}$.

```python
betas = np.linspace(1e-4, 0.02, 1000)
abar = np.cumprod(1 - betas)

def q_sample(x0, t, eps):
    return np.sqrt(abar[t]) * x0 + np.sqrt(1 - abar[t]) * eps

x0 = rng.normal(0, 2, size=200_000); eps = rng.normal(size=200_000)
xt = q_sample(x0, 300, eps)
slope_fit = np.sum(eps * xt) / np.sum(xt ** 2)
print(abar[[0, 299, 999]], slope_fit)
```

</details>

### Exercise 16 · Fréchet Inception Distance
*💻 Code · ★★☆*

FID compares Gaussian fits of feature embeddings of real and generated images:
$\mathrm{FID} = \lVert\mu_1-\mu_2\rVert^2 + \mathrm{Tr}\big(\Sigma_1+\Sigma_2-2(\Sigma_1\Sigma_2)^{1/2}\big)$.
Implement `fid(F1, F2)` taking two feature matrices (rows = images); use `scipy.linalg.sqrtm` and keep the real part.
Then measure FID between `F_real` and (a) a fresh sample of the same distribution, (b) a "mode-collapsed" generator that only produces one cluster.

In [ ]:
def fid(F1, F2):
    return None

def sample_feats(n, collapsed=False):
    centers = np.array([[0, 0, 0], [4, 0, 0], [0, 4, 0]], float)
    k = np.zeros(n, int) if collapsed else rng.integers(0, 3, n)
    return centers[k] + rng.normal(size=(n, 3))

F_real = sample_feats(3000)
fid_same = None
fid_collapsed = None

_a, _b = rng.normal(1, 2, size=(20000, 1)), rng.normal(-1, 0.5, size=(20000, 1))
check('1-D case: (m1-m2)^2 + (s1-s2)^2', None if fid(_a, _b) is None else fid(_a, _b), (_a.mean() - _b.mean())**2 + (_a.std(ddof=1) - _b.std(ddof=1))**2, tol=1e-6)
check('same distribution: small (< 0.1)', None if fid_same is None else fid_same < 0.1, True)
check('mode collapse: large (> 5)', None if fid_collapsed is None else fid_collapsed > 5, True)

<details>
<summary><b>💡 Hint</b></summary>

`mu = F.mean(0)`, `S = np.cov(F, rowvar=False)` (use `np.atleast_2d` for the 1-D case), `sqrtm(S1 @ S2).real`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

In 1-D, $\mathrm{Tr}(\sigma_1^2+\sigma_2^2-2\sigma_1\sigma_2) = (\sigma_1-\sigma_2)^2$. Mode collapse shrinks the covariance and shifts the mean, which FID detects;
it captures both fidelity and diversity, unlike a pure per-sample quality score. In practice the features come from an Inception network and
FID is biased for small sample sizes (compare models with the same $n$).

```python
def fid(F1, F2):
    m1, m2 = F1.mean(0), F2.mean(0)
    S1, S2 = np.atleast_2d(np.cov(F1, rowvar=False)), np.atleast_2d(np.cov(F2, rowvar=False))
    return float(np.sum((m1 - m2) ** 2) + np.trace(S1 + S2 - 2 * sqrtm(S1 @ S2).real))

def sample_feats(n, collapsed=False):
    centers = np.array([[0, 0, 0], [4, 0, 0], [0, 4, 0]], float)
    k = np.zeros(n, int) if collapsed else rng.integers(0, 3, n)
    return centers[k] + rng.normal(size=(n, 3))

F_real = sample_feats(3000)
fid_same = fid(F_real, sample_feats(3000))
fid_collapsed = fid(F_real, sample_feats(3000, collapsed=True))
print(fid_same, fid_collapsed)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Generative Models](Generative%20Models.md).*